# Naive RAG Deep Dive -- Step-by-Step Diagnosis

This notebook walks through the **B1 Naive RAG** retrieval system step by step.
We will:
1. Show how a query is embedded and searched via FAISS
2. Compare retrieved pages against HotpotQA gold supporting facts
3. Compute EM and F1 metrics manually to understand the scoring
4. Run batch evaluation on 10 questions with visualizations
5. Diagnose why this baseline behaves identically to HtmlRAG and HyperRAG

**Key insight:** Naive RAG forms the retrieval backbone for ALL three systems.
Understanding it is essential before diagnosing why they collapse to identical scores.

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is on path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.corpus import load_corpus, load_hotpotqa
from src.embeddings import load_index, search
from src.retrieval import naive_rag, compute_em, compute_f1

# Load data
corpus = load_corpus(PROJECT_ROOT / "data" / "corpus.json")
index, page_ids = load_index(PROJECT_ROOT / "data")
qa_items = load_hotpotqa(split="train", n_samples=50)[:10]

print(f"Corpus: {len(corpus)} pages")
print(f"FAISS index: {index.ntotal} vectors")
print(f"QA items loaded: {len(qa_items)}")

## Step 1: Query Embedding and FAISS Search

Naive RAG works in two stages:
1. **Encode** the query using the same sentence-transformer model (`all-MiniLM-L6-v2`) that was used to build the FAISS index
2. **Search** the FAISS index for the top-k pages with highest cosine similarity

The retrieval is purely based on embedding similarity -- no structural information, no link graph, no HTML context.
Let's see this in action for a single question.

In [ ]:
# Pick the first question
qa = qa_items[0]
question = qa["question"]
answer = qa["answer"]
supporting_facts = qa["supporting_facts"]

print(f"Question: {question}")
print(f"Gold answer: {answer}")
print(f"Supporting facts titles: {list(set(supporting_facts['title']))}")
print()

# Run FAISS search
results = search(question, index, page_ids, corpus, k=5)

print("Top-5 retrieved pages:")
for i, r in enumerate(results):
    print(f"  {i+1}. {r['title']:50s}  score={r['score']:.4f}")

# Horizontal bar chart of similarity scores
titles = [r["title"][:40] for r in results]
scores = [r["score"] for r in results]

fig, ax = plt.subplots(figsize=(10, 4))
y_pos = range(len(titles))
ax.barh(y_pos, scores, color="steelblue")
ax.set_yticks(y_pos)
ax.set_yticklabels(titles)
ax.invert_yaxis()
ax.set_xlabel("Cosine Similarity Score")
ax.set_title(f"FAISS Retrieval Scores for: '{question[:60]}...'")
plt.tight_layout()
plt.show()

## Step 2: Retrieved Pages vs Gold Supporting Facts

HotpotQA provides **supporting_facts** -- the specific Wikipedia pages (and sentence indices) that contain the information needed to answer the question.

For multi-hop questions, there are typically **2 supporting pages** that must be combined to reach the answer. If FAISS retrieval misses one of them, the system cannot answer correctly.

Let's check how many gold pages our top-5 retrieval actually found.

In [ ]:
gold_titles = set(supporting_facts["title"])
retrieved_titles = [r["title"] for r in results]

print("Retrieved vs Gold comparison:")
for title in retrieved_titles:
    match = "[MATCH]" if title in gold_titles else "[     ]"
    print(f"  {match} {title}")

print()
missed = gold_titles - set(retrieved_titles)
if missed:
    print(f"MISSED gold pages ({len(missed)}):")
    for t in missed:
        print(f"  - {t}")
else:
    print("All gold pages were retrieved!")

recall = len(gold_titles & set(retrieved_titles)) / len(gold_titles) if gold_titles else 0
print(f"\nRetrieval recall: {recall:.2f} ({len(gold_titles & set(retrieved_titles))}/{len(gold_titles)} gold pages found)")

## Step 3: EM and F1 Computation Step-by-Step

Our evaluation metrics measure **retrieval quality** (not generation quality):

- **EM (Exact Match):** Does the gold answer string appear (case-insensitive) anywhere in the retrieved context?
- **F1 (Token F1):** What is the token-level overlap between the gold answer and the retrieved context?

Let's compute these manually to understand exactly what happens.

In [ ]:
# Get the full naive_rag context
context = naive_rag(question, index, corpus, k=5)

print(f"Context length: {len(context)} characters")
print(f"First 500 chars:\n{context[:500]}")
print("...")
print()

# Compute metrics
em = compute_em(context, answer)
f1 = compute_f1(context, answer)

print(f"Gold answer: '{answer}'")
print(f"EM score: {em}")
print(f"F1 score: {f1:.4f}")
print()

# Manual EM check
em_manual = answer.lower() in context.lower()
print(f"Manual EM check: '{answer.lower()}' in context.lower() = {em_manual}")
print()

# Manual F1 breakdown
from collections import Counter
pred_tokens = context.lower().split()
gold_tokens = answer.lower().split()
pred_counter = Counter(pred_tokens)
gold_counter = Counter(gold_tokens)
overlap = sum((pred_counter & gold_counter).values())
precision = overlap / len(pred_tokens) if pred_tokens else 0
recall_f1 = overlap / len(gold_tokens) if gold_tokens else 0
manual_f1 = 2 * precision * recall_f1 / (precision + recall_f1) if (precision + recall_f1) > 0 else 0

print("F1 breakdown:")
print(f"  Context tokens: {len(pred_tokens)}")
print(f"  Answer tokens: {len(gold_tokens)} -> {gold_tokens}")
print(f"  Shared tokens (multiset overlap): {overlap}")
print(f"  Precision: {precision:.6f}")
print(f"  Recall: {recall_f1:.4f}")
print(f"  F1: {manual_f1:.4f}")

## Step 4: Batch Evaluation on 10 Questions

Now let's run Naive RAG on all 10 QA items and visualize the results.
We also track **supporting_facts recall** to see how often the gold pages are retrieved.

In [ ]:
results_table = []

for i, qa in enumerate(qa_items):
    q = qa["question"]
    a = qa["answer"]
    sf_titles = set(qa["supporting_facts"]["title"])

    ctx = naive_rag(q, index, corpus, k=5)
    em_val = compute_em(ctx, a)
    f1_val = compute_f1(ctx, a)

    # Supporting facts recall
    retrieved = search(q, index, page_ids, corpus, k=5)
    ret_titles = set(r["title"] for r in retrieved)
    sf_recall = len(sf_titles & ret_titles) / len(sf_titles) if sf_titles else 0

    results_table.append({
        "qid": i, "question": q[:50], "em": em_val, "f1": f1_val,
        "sf_recall": sf_recall
    })

# Print summary table
print(f"{'QID':<5} {'EM':<5} {'F1':<8} {'SF Recall':<10} {'Question':<50}")
print("-" * 78)
for r in results_table:
    print(f"{r['qid']:<5} {r['em']:<5.0f} {r['f1']:<8.4f} {r['sf_recall']:<10.2f} {r['question']}")

avg_em = np.mean([r["em"] for r in results_table])
avg_f1 = np.mean([r["f1"] for r in results_table])
avg_sf = np.mean([r["sf_recall"] for r in results_table])
print(f"\nAverages:  EM={avg_em:.2f}  F1={avg_f1:.4f}  SF_Recall={avg_sf:.2f}")

# Grouped bar chart
x = np.arange(len(results_table))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width/2, [r["em"] for r in results_table], width, label="EM", color="steelblue")
ax.bar(x + width/2, [r["f1"] for r in results_table], width, label="F1", color="coral")

ax.set_xlabel("Question Index")
ax.set_ylabel("Score")
ax.set_title("Naive RAG: EM and F1 per Question (10 questions)")
ax.set_xticks(x)
ax.legend()
plt.tight_layout()
plt.show()

## Diagnosis

### What Naive RAG Does

Naive RAG retrieves pages **purely by embedding similarity** -- no structural or link information is used. The query is encoded with `all-MiniLM-L6-v2`, and FAISS returns the top-k pages with highest cosine similarity.

### Why This Matters for Multi-Hop Questions

HotpotQA is a **multi-hop** dataset. Answering a question typically requires information from **two distinct Wikipedia pages** connected by a reasoning chain. FAISS embedding search may retrieve one relevant page but miss the second because:

- The second page's text is not directly similar to the question
- The bridge entity connecting the two pages is not prominent in the second page's embedding
- The supporting_facts recall metric above shows how often gold pages are actually retrieved

### The Critical Flaw: Shared Retrieval Backbone

This is the **BASELINE** system, but it is also the retrieval backbone for both HtmlRAG and HyperRAG:

1. **HtmlRAG** calls the exact same `search()` function with the same FAISS index -- it only changes the **format** of the context (HTML vs plain text), not the retrieval strategy.
2. **HyperRAG** also starts with the same `search()` call, then attempts graph expansion -- but if expansion finds zero neighbors, the result is identical to Naive RAG.

Any differentiation between the three systems **must happen AFTER the initial FAISS retrieval**. If HtmlRAG only changes formatting and HyperRAG's expansion finds nothing, all three systems produce the same pages and therefore the same EM/F1 scores.

### Next Steps

- See `htmlrag_explained.ipynb` for proof that HTML formatting collapses to identical metrics
- See `hyperrag_explained.ipynb` for proof that graph expansion finds no neighbors